In [13]:
import os
import xarray as xr

# User parameters :
dataset_path = os.path.join(r"~",r"git_folder/torchydra/2024-07-02_merged_simu_sensors_db_saved.nc")
output_dir = 'datasets/demosath_2'
# Define chanels for the dataset
channels = ['simu_AI_WindSpeed', 'simu_V_ST_TrueWindDir', 'simu_V_ST_TrueNacelleDir', 'simu_V_GridRealPowerLog', 'simu_V_MRU_Heave', 'simu_V_MRU_Pitch', 'simu_V_MRU_Roll', 'simu_V_MRU_Longitude_rel', 'simu_V_MRU_Latitude_rel', 'simu_V_RotorRpm', 'simu_V_MRU_Heading', 'simu_V_SPM_LOAD_Pin_1', 'simu_V_SPM_LOAD_Pin_2', 'simu_V_SPM_LOAD_Pin_3', 'simu_V_SPM_LOAD_Pin_4', 'simu_V_SPM_LOAD_Pin_5', 'simu_V_SPM_LOAD_Pin_6'] # 'pitch', 'yaw']
envir_list = ['simu_hs', 'simu_tp', 'simu_dp',  'simu_theta10', 'simu_mag10' ]

# channels which requires a cos / sin decomposition to avoid 360-->0 variations.
heading_angle_vars = ['simu_V_ST_TrueWindDir', 'simu_V_ST_TrueNacelleDir','simu_V_MRU_Heading']

# How to split train / val / test sets.
record_data_input =[
    {'step': 'train', 'start': '2024-01-01 00:00:00', 'end': '2024-05-08 16:00:00'},
    {'step': 'val', 'start': '2024-05-08 16:00:00', 'end': '2024-05-08 18:00:00'},
    # {'step': 'test', 'start': '2024-05-08 14:00:00', 'end': '2024-05-30 23:00:00'},
    {'step': 'test', 'start': '2024-05-08 16:00:00', 'end': '2024-05-08 23:00:00'}
]

df = xr.open_dataset(dataset_path)
variable_list = channels
coordinate_list = list(df.coords)
not_drop_list = coordinate_list + variable_list + envir_list
# drop all variables not in variable_list
df = df.drop_vars([ var for var in df.variables if var not in not_drop_list] )
df = df.dropna(dim='time', how='any')
# Drop Heave higher than 6m
df = df.where(df['simu_V_MRU_Heave'].max(dim='time_sensor') <4, drop=True)
# Drop when the turbine is producing
df = df.where(df['simu_V_RotorRpm'].mean(dim='time_sensor') <3, drop=True)

In [1]:
from ...prepare_data.validity_domain import ValidityDomain

validity_domain = ValidityDomain()

df_training, df_test = validity_domain.find_test_set_in_model_validity_domain(df)

ImportError: attempted relative import with no known parent package